**PROYECTO**       : TOPAZ - PROCESOS DIARIOS  
**NOMBRE**         : nb_ud_egp.ipynb  
**TABLA DESTINO**  : mb_gold_prod.finanzas.fct_egp_diario  
**TABLA FUENTE**   : mb_silver_prod.mmff.h_movimiento_contable  
**OBJETIVO**       : Generar el estado de ganancias y perdidas diario  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Cristopher Castro | MIBANCO | Enith Rodriguez | 2026-09-11 | Creacion de proceso |

## 1. Librerias y dependencias

In [0]:
import logging
import time

from pyspark.sql import functions as F

## 2. Funciones de transformacion

In [0]:
def read_movimientos(tabla, columnas, fecha_proceso):
    """Lee los movimientos contables de la fecha de proceso."""
    return (
        spark.table(tabla)
        .select(*columnas)
        .filter(F.col("fec_proceso") == fecha_proceso)
    )


def add_egp_ajustado(df_egp):
    """Convierte a positivo el resultado de las cuentas de egreso.

    El signo del movimiento depende de la naturaleza de la cuenta y el
    reporte se presenta siempre en valor absoluto.
    """
    return df_egp.withColumn(
        "mto_egp_ajustado",
        F.when(F.col("mto_egp") < 0, F.col("mto_egp") * F.lit(-1))
        .otherwise(F.col("mto_egp")),
    )

## 3. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
dbutils.widgets.text("fechaproceso", "")

var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

logger = logging.getLogger("UD_EGP")
logger.setLevel(logging.INFO)
ini_proceso = time.perf_counter()
logger.info("Inicio del proceso. ambiente=%s fechaproceso=%s",
            var_ambiente, var_fechaproceso)

## 4. Constantes y variables

In [0]:
TBL_MOVIMIENTO_SRC = f"mb_silver_{var_ambiente}.mmff.h_movimiento_contable"
TBL_CUENTA_SRC = f"mb_silver_{var_ambiente}.mmff.m_cuenta_contable"
TBL_EGP_FIN = f"mb_gold_{var_ambiente}.finanzas.fct_egp_diario"

COLUMNAS_MOVIMIENTO = [
    "cod_cuenta_contable",
    "cod_centro_costo",
    "mto_movimiento",
    "fec_proceso",
]

## 5. Logica principal

In [0]:
ini_etapa = time.perf_counter()

df_movimientos = read_movimientos(
    TBL_MOVIMIENTO_SRC, COLUMNAS_MOVIMIENTO, var_fechaproceso
)
df_cuentas = spark.table(TBL_CUENTA_SRC).select(
    "cod_cuenta_contable", "des_cuenta_contable"
)

df_egp = (
    df_movimientos.join(df_cuentas, "cod_cuenta_contable", "left")
    .groupBy("cod_cuenta_contable", "des_cuenta_contable", "cod_centro_costo")
    .agg(F.sum(F.col("mto_movimiento")).alias("mto_egp"))
)

df_egp_final = add_egp_ajustado(df_egp).withColumn(
    "fec_proceso", F.to_date(F.lit(var_fechaproceso))
)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 6. Escritura

In [0]:
try:
    (
        df_egp_final
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_EGP_FIN)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_EGP_FIN, exc)
    raise

logger.info("Fin del proceso. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)